# Cheap-Talk Benchmark — Kaggle Runner

Runs all 7 scenarios (baseline + no_sense + silence + counterfactual + 3 framing variants) for ONE model. Model loads once.

## ⚠ HOW TO GUARANTEE YOUR RESULTS ARE SAVED ⚠

Kaggle has TWO run modes. Only ONE persists your output:

**❌ Interactive run** (clicking Run on each cell):
- `/kaggle/working/` is wiped when the session ends (idle timeout, browser close, etc.)
- You MUST manually download zips from the Output panel before closing

**✅ Save Version → Save & Run All (Commit)** (recommended):
- Top-right button → choose **Save & Run All (Commit)**
- Runs in background — you can close the browser
- When status → `Success`, `/kaggle/working/` is permanently snapshotted as the version output
- Even if it Fails partway, partial outputs are still preserved in the version

Use the second mode. Period.

**Setup:**
1. Right panel → Settings → Accelerator → `GPU T4 x2`
2. Right panel → Settings → Internet → `On`
3. (For gated Llama/Gemma only) Add Kaggle Secret `HF_TOKEN`
4. Pick MODEL in Cell 4
5. Click **Save Version → Save & Run All (Commit)** — and go do something else for ~5 hours

## Cell 1 — Install missing deps (DO NOT touch torch/transformers)

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), "GPU not enabled! Settings → Accelerator → GPU T4 x2"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import transformers
print(f"torch: {torch.__version__}  transformers: {transformers.__version__}")

## Cell 2 — Pull the latest benchmark code

In [ ]:
GITHUB_REPO = "https://github.com/stsimpe/cheaptalk_bench.git"

import os
if os.path.exists('/kaggle/working/repo'):
    %cd /kaggle/working/repo
    !git pull
else:
    !git clone $GITHUB_REPO /kaggle/working/repo

REPO_CODE = '/kaggle/working/repo/cheaptalk_bench'
if not os.path.exists(REPO_CODE):
    REPO_CODE = '/kaggle/working/repo'
%cd $REPO_CODE
print('Working dir:', os.getcwd())
!ls run_all_scenarios.py llm_client.py agent.py config.py 2>/dev/null

## Cell 3 — HF token (only for gated Llama / Gemma)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded')
except Exception as e:
    print(f'No HF_TOKEN (fine for Qwen): {e}')

## Cell 4 — Pick model + define absolute output paths

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
# MODEL = "Qwen/Qwen2.5-3B-Instruct"
# MODEL = "Qwen/Qwen2.5-14B-Instruct"
# MODEL = "google/gemma-2-9b-it"             # gated -- needs HF_TOKEN + accepted license
# MODEL = "meta-llama/Llama-3.1-8B-Instruct"  # gated -- needs Meta approval

MODEL_SHORT = MODEL.split('/')[-1]
BASE_DIR = f'/kaggle/working/results/{MODEL_SHORT}'
os.makedirs(BASE_DIR, exist_ok=True)

print(f"Model       : {MODEL}")
print(f"Output base : {BASE_DIR}")
print(f"Zip mirror  : /kaggle/working/  (per-scenario zips appear here as they finish)")

## Cell 5 — Smoke test (5-10 min, all 7 scenarios at 2x8)

**Skip this cell** if you're confident — saves 10 min on full sweep.

In [ ]:
!python run_all_scenarios.py --provider local --model-id $MODEL \
    --out-dir-base $BASE_DIR/_smoke --zip-mirror /kaggle/working/_smoke_zips \
    --quick --no-probe

## Cell 6 — Full sweep, all 7 scenarios in one process (~5-6 h)

Each scenario gets its own folder AND its own zip in `/kaggle/working/`. Even if the run crashes partway, every completed scenario is already zipped and saved.

In [ ]:
!python run_all_scenarios.py --provider local --model-id $MODEL \
    --out-dir-base $BASE_DIR \
    --zip-mirror /kaggle/working \
    --no-probe

## Cell 7 — ✅ Verify what got saved (run this BEFORE you assume anything)

Lists every JSON, every zip, every folder. If this prints zero files, something is wrong and you need to investigate before the session ends.

In [ ]:
import os, glob, json

print('=' * 70)
print(f'VERIFICATION REPORT — {MODEL_SHORT}')
print('=' * 70)

print('\n--- Per-scenario folders under BASE_DIR ---')
scenarios = ['baseline', 'no_sense', 'silence', 'counterfactual',
             'framing_business', 'framing_team', 'framing_competitive']
for sc in scenarios:
    sc_dir = f'{BASE_DIR}/{sc}'
    if os.path.exists(sc_dir):
        files = glob.glob(f'{sc_dir}/**/*.json', recursive=True)
        print(f'  {sc:24s}  {len(files):3d} JSONs')
    else:
        print(f'  {sc:24s}  MISSING')

print('\n--- Zip files in /kaggle/working/ (visible in Output panel) ---')
zips = sorted(glob.glob('/kaggle/working/*.zip'))
for z in zips:
    size_mb = os.path.getsize(z) / 1e6
    print(f'  {os.path.basename(z):40s}  {size_mb:6.2f} MB')
if not zips:
    print('  (none -- something went wrong)')

print('\n--- Progress log ---')
progress_path = f'{BASE_DIR}/zips/_progress.json'
if os.path.exists(progress_path):
    with open(progress_path) as f:
        prog = json.load(f)
    print(f'  Model: {prog.get("model")}')
    print(f'  Started: {prog.get("started_at")}')
    print(f'  Session tokens: {prog.get("session_tokens", 0):,}')
    print(f'  Scenarios done: {[s["name"] for s in prog.get("scenarios_done", [])]}')
    if prog.get('current_scenario'):
        print(f'  Currently running: {prog["current_scenario"]}')

total_json = len(glob.glob(f'{BASE_DIR}/**/*.json', recursive=True))
print(f'\n--- TOTAL: {total_json} JSON files under {BASE_DIR} ---')

## Cell 8 — Final combined zip (one zip with everything)

Useful if you want a single download instead of 7 per-scenario zips.

In [ ]:
import shutil, os
if os.path.exists(BASE_DIR):
    combined = f'/kaggle/working/all_{MODEL_SHORT}'
    shutil.make_archive(combined, 'zip', BASE_DIR)
    size_mb = os.path.getsize(combined + '.zip') / 1e6
    print(f'Combined zip: {combined}.zip  ({size_mb:.2f} MB)')
else:
    print(f'BASE_DIR does not exist: {BASE_DIR}')
!ls -lh /kaggle/working/*.zip

## Cell 9 — Per-scenario analysis (optional preview)

In [ ]:
import os
scenarios = ['baseline', 'no_sense', 'silence', 'counterfactual',
             'framing_business', 'framing_team', 'framing_competitive']
for scenario in scenarios:
    sc_dir = f'{BASE_DIR}/{scenario}'
    if not os.path.exists(sc_dir):
        continue
    print(f'\n{"=" * 70}')
    print(f'  {scenario}')
    print('=' * 70)
    !python analysis.py --results-dir $sc_dir 2>&1 | tail -30

## Cell 10 — Recovery diagnostic (only if Cell 7 looks wrong)

In [ ]:
import glob, os
print('=== Every JSON under /kaggle/working/ ===')
jsons = glob.glob('/kaggle/working/**/*.json', recursive=True)
for f in jsons[:30]:
    print(f' ', f, f'({os.path.getsize(f)//1024} KB)')
print(f'  ... total: {len(jsons)}')
print('\n=== Everything in /kaggle/working/ ===')
!find /kaggle/working/ -maxdepth 2 -type f -o -maxdepth 2 -type d | head -40